# 09 — Consolidación y submission

**Objetivo.** Recoger el mejor enfoque de cada índice, verificar compatibilidad de shapes, generar y validar la submission final.

**Decisión tomada.** Un único notebook de consolidación que lee los 6 JSON de `results/` y reconstruye las predicciones. Generación parametrizada para adaptarse al formato exacto que indique el profesor.

**Qué hace.** Cargar 6 JSON, tabla comparativa de RMSE, generar predicciones según `approach_type`, `generar_submission` + `validar_submission`, sección de FALLBACK MANUAL.

**Qué NO hace.** No entrena nada. No modifica los JSON ni los modelos.

**Inputs.** `results/index_A.json` … `results/index_F.json`, todos los CSV de `data/`, modelos en `models/`

**Output esperado.** `results/submission.csv` — 252 filas × 6 columnas.

---

### Convención de ancla (CRÍTICO — una sola fuente de verdad)

El precio inicial para reconstruir precios desde log-retornos es **siempre** `train_indices[índice].iloc[-1]` — el último precio real conocido de train. Este valor se pasa como `precio_inicial` a `predict_autoregressive`. Todos los notebooks de índice usan esta misma convención internamente a través de `backtest_autoregressive`. **Nunca** se usa un precio del período de test como ancla.

## 0. GPU workaround + imports

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data,
    predict_autoregressive, precios_a_logret,
    baseline_flat, baseline_drift, baseline_random_walk,
    generar_submission, validar_submission,
    plot_rmse_by_index,
    DATA_DIR, N_DAYS_PRED, V_IN_SHARED, INDEX_COLS
)

## 1. Cargar los 6 JSON de results/

Si falta alguno, el proceso se detiene con un mensaje claro. **No subir sin los 6 JSON.**

In [ ]:
INDEX_LABELS = ['A', 'B', 'C', 'D', 'E', 'F']
infos = {}

for lbl in INDEX_LABELS:
    path = f'results/index_{lbl}.json'
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Falta {path} — ejecutar primero el notebook 0{3+INDEX_LABELS.index(lbl)}_index_{lbl}.ipynb'
        )
    with open(path) as f:
        infos[f'Index_{lbl}'] = json.load(f)

print(f'[OK] {len(infos)} JSONs cargados correctamente')

## 2. Tabla comparativa de RMSE por índice

In [ ]:
rows = []
for col, info in infos.items():
    rows.append({
        'index':         col,
        'approach_type': info['approach_type'],
        'strategy':      info['strategy'],
        'owner':         info['owner'],
        'rmse_backtest': info['rmse_backtest_252d'],
    })

tabla = pd.DataFrame(rows).set_index('index').sort_values('rmse_backtest')
print(tabla.to_string())
print(f'\nRMSE medio estimado: {tabla["rmse_backtest"].mean():,.0f}')

# Plot
rmse_dict = {r['index']: r['rmse_backtest'] for r in rows}
plot_rmse_by_index(rmse_dict, title='RMSE backtest 252d por índice — enfoque elegido')

## 3. Carga de datos para reconstruir predicciones

In [ ]:
data = load_hackathon_data(DATA_DIR)
idx  = data['train_indices']

## 4. Generar predicciones para los 252 días de producción

Switch sobre `approach_type` para cada índice:
- `nn` / `nn_ensemble`: cargar modelo de `model_path`, construir ventana inicial con los últimos `v_in` log-retornos de train, ancla = `train_indices[col].iloc[-1]`, llamar `predict_autoregressive`.
- `ghost`: aplicar `ghost_predict_252` (misma lógica que en `06_index_D`) con `ghost_source_index` y `ghost_lag`.
- `baseline_flat/drift/rw`: llamar la función correspondiente sobre la serie de train.

In [ ]:
import keras
from utils import baseline_flat, baseline_drift, baseline_random_walk, precios_a_logret

predicciones = {}

for col, info in infos.items():
    serie    = idx[col].dropna().values
    at       = info['approach_type']

    # ── Ancla: último precio real de train — convención única ──────────────────
    precio_inicial = float(idx[col].iloc[-1])

    if at in ('nn', 'nn_ensemble'):
        model      = keras.models.load_model(info['model_path'])
        predict_fn = lambda x, m=model: m.predict(x, verbose=0).ravel()[0]
        v_in       = info['v_in']

        # Ventana inicial: últimos v_in log-retornos de train
        log_rets = precios_a_logret(serie).astype('float32')
        ventana  = log_rets[-v_in:].reshape(v_in, 1)

        # predict_autoregressive reconstruye precios usando precio_inicial como ancla
        preds = predict_autoregressive(
            predict_fn, ventana, N_DAYS_PRED, precio_inicial=precio_inicial
        )

    elif at == 'ghost':
        src  = info['ghost_source_index']
        lag  = info['ghost_lag']
        fuente_train = idx[src].dropna().values

        # Primeros `lag` días: precios ya conocidos del fuente (no es leakage)
        preds_ghost = list(fuente_train[-lag:]) if lag > 0 else []
        # Días restantes: baseline_drift del fuente como proxy
        n_rest = N_DAYS_PRED - lag
        if n_rest > 0:
            preds_ghost.extend(baseline_drift(fuente_train, n_rest))
        preds = np.array(preds_ghost[:N_DAYS_PRED])

    elif at == 'baseline_flat':
        preds = baseline_flat(serie, N_DAYS_PRED)

    elif at == 'baseline_drift':
        preds = baseline_drift(serie, N_DAYS_PRED)

    elif at == 'baseline_rw':
        preds = baseline_random_walk(serie, N_DAYS_PRED)

    else:
        raise ValueError(f'approach_type desconocido para {col}: {at}')

    predicciones[col] = preds
    print(f'  {col}: {len(preds)} predicciones  min={preds.min():.0f}  max={preds.max():.0f}')

## 5. Verificación de shapes

In [ ]:
df_pred = pd.DataFrame(predicciones, columns=INDEX_COLS)

assert df_pred.shape == (N_DAYS_PRED, 6), \
    f'Shape incorrecto: {df_pred.shape} (esperado: ({N_DAYS_PRED}, 6))'
assert not df_pred.isnull().any().any(), \
    f'NaN detectados:\n{df_pred.isnull().sum()}'

print(f'[OK] df_pred shape: {df_pred.shape}  — sin NaN')
print(df_pred.describe().round(0))

## 6. Generación parametrizada de submission

El profesor confirmará el formato exacto el sábado (Excel, CSV, separador, decimales). Ajustar los parámetros aquí y ejecutar. `generar_submission` requiere `test_dates` para el índice de fechas del CSV.

In [ ]:
# ── Parámetros de formato — ajustar según instrucciones del profesor ──────────
SUBMISSION_PATH = 'results/submission.csv'
DECIMAL_PLACES  = 4        # nº de decimales en el CSV
SEPARATOR       = ','      # separador de columnas (por defecto CSV estándar)
INCLUDE_DATE    = True     # True = primera columna = Date; False = solo valores
INCLUDE_HEADER  = True     # True = cabecera con nombres de columna

# generar_submission produce el CSV con índice de fechas
test_dates = data.get('test_dates')   # None si no disponible aún

if test_dates is None:
    print('[AVISO] test_dates.csv no disponible — el CSV se generará sin columna Date')
    # Construir fechas ficticias para no bloquear la generación
    test_dates = pd.DataFrame(index=pd.RangeIndex(N_DAYS_PRED))

df_sub = generar_submission(predicciones, test_dates, filepath=SUBMISSION_PATH)

# Si el profesor pide formato diferente: re-exportar df_sub manualmente
if not INCLUDE_DATE:
    df_sub.reset_index(drop=True).to_csv(
        SUBMISSION_PATH, sep=SEPARATOR,
        float_format=f'%.{DECIMAL_PLACES}f',
        header=INCLUDE_HEADER, index=False
    )
else:
    df_sub.to_csv(
        SUBMISSION_PATH, sep=SEPARATOR,
        float_format=f'%.{DECIMAL_PLACES}f',
        header=INCLUDE_HEADER
    )
print(f'Submission escrita en: {SUBMISSION_PATH}')

## 7. Validación de submission

**OBLIGATORIO antes de cualquier entrega.** `validar_submission` comprueba dimensiones, columnas, NaN/Inf y opcionalmente fechas.

In [ ]:
ok = validar_submission(SUBMISSION_PATH)
if not ok:
    raise RuntimeError('Submission NO válida — corregir antes de subir')
print('\n[LISTO] La submission ha pasado la validación — seguro subir')

## 8. FALLBACK MANUAL — si la generación automática falla

Si cualquier celda anterior falla y no hay tiempo de depurar, copiar los valores directamente al Excel del profesor.

**Método principal — pegar en Excel con columnas separadas:**

In [ ]:
# MÉTODO PRINCIPAL: copia df_pred al portapapeles listo para pegar en Excel
# Cada columna va a su celda correcta (separadas por tabulador)
df_pred.to_clipboard(index=False)
print('[PORTAPAPELES] df_pred copiado — pegar (Ctrl+V) en la celda A1 del Excel del profesor')
print('Columnas:', list(df_pred.columns))
print('Filas:', len(df_pred))

In [ ]:
# MÉTODO DE RESPALDO: imprimir CSV en la salida para copiar/pegar a mano
print('=== VALORES EN FORMATO CSV (copiar todo y pegar) ===')
print(df_pred.to_csv(index=False, float_format=f'%.{DECIMAL_PLACES}f'))